# Standard problem 4

## Problem specification

The sample is a thin film cuboid with dimensions:

- length $l_{x} = 500 \,\text{nm}$,
- width $l_{y} = 125 \,\text{nm}$, and
- thickness $l_{z} = 3 \,\text{nm}$.

The material parameters (similar to permalloy) are:

- exchange energy constant $A = 1.3 \times 10^{-11} \,\text{J/m}$,
- magnetisation saturation $M_\text{s} = 8 \times 10^{5} \,\text{A/m}$.

Magnetisation dynamics are governed by the Landau-Lifshitz-Gilbert equation

$$\frac{d\mathbf{m}}{dt} = \underbrace{-\gamma_{0}(\mathbf{m} \times \mathbf{H}_\text{eff})}_\text{precession} + \underbrace{\alpha\left(\mathbf{m} \times \frac{d\mathbf{m}}{dt}\right)}_\text{damping}$$

where $\gamma_{0} = 2.211 \times 10^{5} \,\text{m}\,\text{A}^{-1}\,\text{s}^{-1}$ and Gilbert damping $\alpha=0.02$.

In the standard problem 4, the system is first relaxed at zero external magnetic field and then, starting from the obtained equlibrium configuration, the magnetisation dynamics are simulated for two external magnetic fields $\mathbf{B}_{1} = (-24.6, 4.3, 0.0) \,\text{mT}$ and $\mathbf{B}_{2} = (-35.5, -6.3, 0.0) \,\text{mT}$.

More detailed specification of Standard problem 4 can be found in Ref. 1.

## Simulation


In the first step, we import the required modules.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import neuralmag as nm

mu0 = 4 * np.pi * 1e-7

Now, we can set all required geometry and material parameters.

In [ ]:
# Geometry
lx = 500e-9  # x dimension of the sample(m)
ly = 125e-9  # y dimension of the sample (m)
lz = 3e-9  # sample thickness (m)

# Material (permalloy) parameters
Ms = 8e5  # saturation magnetisation (A/m)
A = 1.3e-11  # exchange energy constant (J/m)

# Dynamics (LLG equation) parameters
gamma0 = 2.211e5  # gyromagnetic ratio (m/As)
alpha = 0.02  # Gilbert damping

## First stage

In the first stage, we need to relax the system at zero external magnetic field.

In order to completely define the micromagnetic system, we need to provide:

1. energy $E$
2. dynamics $\text{d}\mathbf{m}/\text{d}t$
3. magnetisation $\mathbf{m}$

The mesh is created by providing the size and number of discretisation cell. We choose the discretisation to be $(5, 5, 3) \,\text{nm}$.

In [ ]:
mesh = nm.Mesh([100, 25, 1], [5e-09, 5e-09, 3e-09], [0.0, 0.0, 0.0])
state = nm.State(mesh)

**Hamiltonian:** In the second step, we define the system's Hamiltonian. In this standard problem, the Hamiltonian contains only exchange and demagnetisation energy terms. Please note that in the first simulation stage, there is no applied external magnetic field. Therefore, we do not add Zeeman energy term to the Hamiltonian.

In [ ]:
state.material.Ms = 800000.0
state.material.A = 1.3e-11
h_ext = nm.VectorFunction(state).fill((0, 0, 0))

**Dynamics:** Similarly, the system's dynamics is defined by providing precession and damping (alpha) terms (LLG equation).

In [ ]:
state.material.alpha = 1

**Magnetisation:** Finally, we have to provide the magnetisation configuration that is going to be relaxed subsequently. We choose the uniform configuration in $(1, 0.25, 0.1)$ direction, and as norm (magnitude) we set the magnetisation saturation $M_\text{s}$.

In [ ]:
state.m = nm.VectorFunction(state).fill((1, 0.25, 0.1))

We then need to register the energy terms.

In [ ]:
nm.ExchangeField().register(state, "exchange")
nm.DemagField().register(state, "demag")
nm.ExternalField(h_ext).register(state, "external")
nm.TotalField("exchange", "demag", "external").register(state)

Now, the system is fully defined.

**Energy minimisation:** The state (its magnetisation) is evolved using the LLG equation. In the first stage, we need to relax the system - minimise its energy.

In [ ]:
llg = nm.LLGSolver(state)
llg.step(10e-10)

We can now obtain some numerical data characteristic to the magnetisation field as is useful for standard problem 4:

In [ ]:
# Add Zeeman energy term to the Hamiltonian
H1 = (-24.6e-3 / mu0, 4.3e-3 / mu0, 0.0)
state.h_external = nm.VectorFunction(state).fill(H1)

Finally, we can run the simulation using the LLGSolver. We run the magnetisation evolution for $t=1 \,\text{ns}$, during which we save the system's state $n=100$ times.

In [ ]:
llg = nm.LLGSolver(state)

logger = nm.Logger("data", ["t", "m"], ["m"])
for _ in range(100):
    llg.step(1.0e-11)
    logger.log(state)

### Postprocessing

Finally, we want to plot the average magnetisation configuration `my` as a function of time `t`:

In [ ]:
data = np.genfromtxt(
    "data/log.dat", delimiter="    ", comments="#", names=["t", "m_x", "m_y", "m_z"]
)

In [ ]:
plt.plot(data["t"], data["m_y"], label="neural_mag")
plt.legend()
plt.xlabel("t (s)")
plt.ylabel("Average my (normalised)")

We can plot another snap shot of the magnetisation at the end of the run, which shows the fluctuations in the simulation at this time:

## References

[1] µMAG Site Directory: http://www.ctcms.nist.gov/~rdm/mumag.org.html
    
This tutorial was adapted from [Ubermag](https://ubermag.github.io/).